# LeRobot v2.1 데이터셋 병합 파이프라인

여러 Hugging Face **dataset** repo를 받은 뒤, **에피소드 인덱스·전역 `index`·비디오 경로**를 다시 매겨 **하나의 v2.1 데이터셋**으로 합칩니다. v3 변환 없음.

**순서**

1. 설정 (`SOURCE_REPO_IDS`, 로컬 병합 폴더, Hub 푸시 옵션)
2. 소스별 `snapshot_download` (repo 순차 — tqdm·중첩 스레드 충돌 방지)
3. `meta/info.json` 등 호환 검사 (동일 `robot_type`, `fps`, feature shape)
4. 병합: `data/`, `videos/`, `meta/episodes.jsonl`, `meta/episodes_stats.jsonl`, `meta/info.json`, `meta/stats.json`
5. (선택) `HfApi.upload_folder`로 Hub 업로드 (`LeRobotDataset` 미사용)

**주의**

- 소스별 `tasks.jsonl` 내용이 같아야 합니다 (다르면 검사에서 중단).
- 병합 순서는 설정 리스트 순서입니다 (기본: trial3 → trial4 → trial5).


In [1]:
from pathlib import Path

# --- 편집 구간: 소스 HF dataset repo_id (병합 순서 = 리스트 순서) ---
SOURCE_REPO_IDS = [
    "learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof",
    "learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof",
    "learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof",
]

# 로컬 출력 폴더명 (dataset_base 아래)
MERGE_LOCAL_DIR = "ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial345_27dof_merge"

# 소스 다운로드 캐시: dataset_base / MERGE_STAGE_DIR / <repo 마지막 세그먼트>
MERGE_STAGE_DIR = "_merge_sources_trial345"

DATASET_BASE = Path("/workspace/dataset").resolve()
MERGE_ROOT = (DATASET_BASE / MERGE_LOCAL_DIR).resolve()
STAGING_ROOT = (DATASET_BASE / MERGE_STAGE_DIR).resolve()

SKIP_DOWNLOAD = False
MERGE_CLEAR = True  # True면 MERGE_ROOT 를 삭제 후 다시 병합
SKIP_MERGE = False

# Hub (v2.1 그대로 업로드 — LeRobotDataset 미사용)
PUSH_TO_HUB = True
PUSH_REPO_ID = "learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial345_27dof_merge"
PUSH_PRIVATE = False


def _staging_dir_for_repo(repo_id: str) -> Path:
    return (STAGING_ROOT / repo_id.split("/")[-1]).resolve()


print("DATASET_BASE", DATASET_BASE)
print("STAGING_ROOT", STAGING_ROOT)
print("MERGE_ROOT", MERGE_ROOT)
print("소스 개수:", len(SOURCE_REPO_IDS))
for r in SOURCE_REPO_IDS:
    print(" ", r, "->", _staging_dir_for_repo(r))
print("PUSH_TO_HUB", PUSH_TO_HUB, PUSH_REPO_ID if PUSH_TO_HUB else "")


DATASET_BASE /workspace/dataset
STAGING_ROOT /workspace/dataset/_merge_sources_trial345
MERGE_ROOT /workspace/dataset/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial345_27dof_merge
소스 개수: 3
  learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof
  learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof
  learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof
PUSH_TO_HUB True learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial345_27dof_merge


## 1) 소스 다운로드

각 repo를 `STAGING_ROOT/<repo 마지막 세그먼트>`에 **순차** `snapshot_download` 합니다.


In [2]:
if SKIP_DOWNLOAD:
    print("[skip] 다운로드")
else:
    from huggingface_hub import snapshot_download

    STAGING_ROOT.mkdir(parents=True, exist_ok=True)
    # repo별 순차 다운로드: 외부 ThreadPoolExecutor 와 snapshot_download 내부
    # thread_map + tqdm 이 겹치면 Jupyter 환경에서 tqdm._lock AttributeError 가 날 수 있음.
    for rid in SOURCE_REPO_IDS:
        dest = _staging_dir_for_repo(rid)
        dest.mkdir(parents=True, exist_ok=True)
        print("시작:", rid, "->", dest)
        snapshot_download(
            repo_id=rid,
            repo_type="dataset",
            local_dir=str(dest),
        )
        print("[ok] 다운로드:", rid)
    print("[ok] 전체 소스 다운로드 완료")


시작: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 48 files: 100%|██████████| 48/48 [00:00<00:00, 1452.88it/s]


[ok] 다운로드: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial3_27dof
시작: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof


Fetching 48 files: 100%|██████████| 48/48 [00:00<00:00, 1429.91it/s]


[ok] 다운로드: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial4_27dof
시작: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof -> /workspace/dataset/_merge_sources_trial345/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof


Fetching 48 files: 100%|██████████| 48/48 [00:03<00:00, 14.16it/s]

[ok] 다운로드: learner1119/ffw_sh5_rev1_ffw_sh5_rev1_20260503_trial5_27dof
[ok] 전체 소스 다운로드 완료


## 2) v2.1 호환 검사

`codebase_version`, `fps`, `robot_type`, `observation.state`/`action` shape·이름이 소스 간 동일한지 확인합니다.


In [3]:
import json
from pathlib import Path


def _load_info(root: Path) -> dict:
    p = root / "meta" / "info.json"
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def _info_fingerprint(info: dict) -> dict:
    keys = (
        "codebase_version",
        "robot_type",
        "fps",
        "chunks_size",
        "data_path",
        "video_path",
    )
    out = {k: info.get(k) for k in keys}
    for feat in ("observation.state", "action"):
        f = info.get("features", {}).get(feat, {})
        out[f"{feat}.shape"] = list(f.get("shape", []))
        out[f"{feat}.names"] = list(f.get("names", []))
    return out


def _read_tasks_jsonl(root: Path) -> list[dict]:
    p = root / "meta" / "tasks.jsonl"
    rows = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


SOURCE_ROOTS = [_staging_dir_for_repo(r) for r in SOURCE_REPO_IDS]
for s in SOURCE_ROOTS:
    if not s.is_dir():
        raise FileNotFoundError(f"스테이징 없음: {s}\n다운로드 셀을 실행하세요.")

infos = [_load_info(s) for s in SOURCE_ROOTS]
fp0 = _info_fingerprint(infos[0])
for i, inf in enumerate(infos[1:], start=2):
    fp = _info_fingerprint(inf)
    if fp != fp0:
        raise RuntimeError(f"info 불일치: 소스1 vs 소스{i}\n{fp0}\nvs\n{fp}")

tasks0 = _read_tasks_jsonl(SOURCE_ROOTS[0])
for i, s in enumerate(SOURCE_ROOTS[1:], start=2):
    t = _read_tasks_jsonl(s)
    if t != tasks0:
        raise RuntimeError(f"tasks.jsonl 불일치: 소스1 vs 소스{i}")

if infos[0].get("codebase_version") != "v2.1":
    raise RuntimeError("v2.1 만 지원합니다. codebase_version=" + str(infos[0].get("codebase_version")))

print("[ok] 호환 검사 통과 (v2.1, 동일 스키마·tasks)")


[ok] 호환 검사 통과 (v2.1, 동일 스키마·tasks)


## 3) 병합 (v2.1 유지)

- `episodes.jsonl` 순서대로 에피소드를 이어 붙이며, 새 `episode_index`는 0 … N-1.
- 각 행의 `index`는 병합 데이터셋 기준 전역 프레임 인덱스.
- `videos/`가 있으면 에피소드 번호·chunk 경로에 맞게 복사.
- `episodes_stats.jsonl`: 테이블에서 구할 수 있는 항목은 갱신, `observation.images.*`는 **원본 소스 에피소드 통계 복사**.


In [4]:
import copy
import json
import shutil
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from lerobot.datasets.utils import STATS_PATH, write_json


def _list_episode_parquets(root: Path) -> list[tuple[int, Path]]:
    out: list[tuple[int, Path]] = []
    for p in sorted(root.glob("data/chunk-*/episode_*.parquet")):
        stem = p.stem
        idx = int(stem.split("_", 1)[1])
        out.append((idx, p))
    out.sort(key=lambda x: x[0])
    return out


def _load_episodes_jsonl(root: Path) -> list[dict]:
    rows = []
    with open(root / "meta" / "episodes.jsonl", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _load_episodes_stats_map(root: Path) -> dict[int, dict]:
    m: dict[int, dict] = {}
    p = root / "meta" / "episodes_stats.jsonl"
    if not p.is_file():
        return m
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            m[int(row["episode_index"])] = row
    return m


def _scalar_stats_from_array(arr: np.ndarray) -> dict:
    arr = np.asarray(arr, dtype=np.float64).ravel()
    n = int(arr.size)
    if n == 0:
        raise ValueError("empty array")
    return {
        "min": [float(arr.min())],
        "max": [float(arr.max())],
        "mean": [float(arr.mean())],
        "std": [float(arr.std())],
        "count": [n],
        "q01": [float(np.quantile(arr, 0.01))],
        "q99": [float(np.quantile(arr, 0.99))],
    }


def _vector_stats_from_table(table, key: str, dim: int) -> dict:
    n = table.num_rows
    rows = []
    for i in range(n):
        v = np.asarray(table[key][i].as_py(), dtype=np.float64).ravel()
        if v.size != dim:
            raise ValueError(f"{key}: row {i} len {v.size} != {dim}")
        rows.append(v)
    X = np.stack(rows, axis=0)
    return {
        "min": X.min(axis=0).tolist(),
        "max": X.max(axis=0).tolist(),
        "mean": X.mean(axis=0).tolist(),
        "std": X.std(axis=0).tolist(),
        "count": [int(n)],
    }


def _build_episode_stats_row(
    new_ep: int,
    table,
    global_index_start: int,
    src_template: dict | None,
    state_dim: int,
) -> dict:
    n = table.num_rows
    base = copy.deepcopy(src_template) if src_template else {"stats": {}}
    base["episode_index"] = int(new_ep)
    st = base.setdefault("stats", {})

    for col in ("timestamp", "frame_index", "task_index"):
        if col in table.column_names:
            arr = np.asarray(table[col].to_pylist(), dtype=np.float64)
            st[col] = _scalar_stats_from_array(arr)

    st["episode_index"] = {
        "min": [float(new_ep)],
        "max": [float(new_ep)],
        "mean": [float(new_ep)],
        "std": [0.0],
        "count": [n],
        "q01": [float(new_ep)],
        "q99": [float(new_ep)],
    }
    idx_arr = np.arange(global_index_start, global_index_start + n, dtype=np.float64)
    st["index"] = _scalar_stats_from_array(idx_arr)

    for key in ("observation.state", "action"):
        if key in table.column_names:
            st[key] = _vector_stats_from_table(table, key, state_dim)

    if src_template and "stats" in src_template:
        for k, v in src_template["stats"].items():
            if k.startswith("observation.images") and k not in st:
                st[k] = copy.deepcopy(v)

    return base


def _list_episode_videos(src_root: Path, old_ep: int) -> list[Path]:
    return sorted(src_root.glob(f"videos/**/episode_{old_ep:06d}.mp4"))


def _copy_episode_videos(
    src_root: Path,
    merge_root: Path,
    old_ep: int,
    new_ep: int,
    chunks_size: int,
) -> None:
    for src in _list_episode_videos(src_root, old_ep):
        rel = src.relative_to(src_root / "videos")
        if len(rel.parts) < 2:
            continue
        video_key = rel.parts[1]
        chunk_new = new_ep // chunks_size
        dst = merge_root / "videos" / f"chunk-{chunk_new:03d}" / video_key / f"episode_{new_ep:06d}.mp4"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)


def _list_v21_data_parquets(dataset_root: Path) -> list[Path]:
    paths = sorted(dataset_root.glob("data/chunk-*/episode_*.parquet"))
    if paths:
        return paths
    alt = sorted(dataset_root.glob("data/chunk-*/*.parquet"))
    return [p for p in alt if p.name.startswith("episode_")]


def _recompute_global_vector_stats(
    dataset_root: Path, keys: list[str], dim: int
) -> dict[str, dict]:
    out: dict[str, dict] = {}
    chunks: dict[str, list] = {k: [] for k in keys}
    for fp in _list_v21_data_parquets(dataset_root):
        t = pq.read_table(fp, columns=keys)
        n = len(t)
        for k in keys:
            for i in range(n):
                arr = np.asarray(t[k][i].as_py(), dtype=np.float64)
                if arr.size != dim:
                    raise ValueError(f"{fp} {k} row {i}: len {arr.size} != {dim}")
                chunks[k].append(arr)
    for k in keys:
        X = np.stack(chunks[k], axis=0)
        out[k] = {
            "min": X.min(axis=0).tolist(),
            "max": X.max(axis=0).tolist(),
            "mean": X.mean(axis=0).tolist(),
            "std": X.std(axis=0).tolist(),
            "count": [int(X.shape[0])],
        }
    return out


def _recompute_global_scalar_stats(dataset_root: Path, cols: list[str]) -> dict[str, dict]:
    arrs: dict[str, list] = {c: [] for c in cols}
    for fp in _list_v21_data_parquets(dataset_root):
        table = pq.read_table(fp)
        have = [c for c in cols if c in table.column_names]
        if not have:
            continue
        t = table.select(have)
        for c in have:
            arrs[c].append(np.asarray(t[c].to_pylist(), dtype=np.float64))
    out: dict[str, dict] = {}
    for c in cols:
        if not arrs[c]:
            continue
        a = np.concatenate(arrs[c])
        out[c] = _scalar_stats_from_array(a)
    return out


if SKIP_MERGE:
    print("[skip] 병합")
else:
    ref = _load_info(SOURCE_ROOTS[0])
    chunks_size = int(ref.get("chunks_size", 1000))
    state_dim = int(ref["features"]["observation.state"]["shape"][0])

    if MERGE_CLEAR and MERGE_ROOT.exists():
        shutil.rmtree(MERGE_ROOT)
    MERGE_ROOT.mkdir(parents=True, exist_ok=True)
    (MERGE_ROOT / "meta").mkdir(parents=True, exist_ok=True)
    (MERGE_ROOT / "data").mkdir(parents=True, exist_ok=True)

    global_ep = 0
    global_frame = 0
    merged_episodes_lines: list[dict] = []
    merged_stats_lines: list[dict] = []

    for src in SOURCE_ROOTS:
        ep_rows = _load_episodes_jsonl(src)
        ep_stats_map = _load_episodes_stats_map(src)
        ep_paths = {i: p for i, p in _list_episode_parquets(src)}

        for ep_row in ep_rows:
            old_ep = int(ep_row["episode_index"])
            if old_ep not in ep_paths:
                raise FileNotFoundError(f"{src}: episode_{old_ep:06d}.parquet 없음")
            src_parquet = ep_paths[old_ep]
            table = pq.read_table(src_parquet)
            n = table.num_rows
            new_ep = global_ep

            new_episode_index = pa.array([new_ep] * n, type=pa.int64())
            new_index = pa.array(
                [global_frame + i for i in range(n)], type=pa.int64()
            )

            names = list(table.column_names)
            arrays = []
            for name in names:
                if name == "episode_index":
                    arrays.append(new_episode_index)
                elif name == "index":
                    arrays.append(new_index)
                else:
                    arrays.append(table[name])
            new_table = pa.Table.from_arrays(arrays, names=names)

            chunk_dir = new_ep // chunks_size
            out_pq = MERGE_ROOT / "data" / f"chunk-{chunk_dir:03d}" / f"episode_{new_ep:06d}.parquet"
            out_pq.parent.mkdir(parents=True, exist_ok=True)
            pq.write_table(new_table, out_pq)

            tpl = ep_stats_map.get(old_ep)
            merged_stats_lines.append(
                _build_episode_stats_row(
                    new_ep, new_table, global_frame, tpl, state_dim
                )
            )

            merged_episodes_lines.append(
                {
                    "episode_index": new_ep,
                    "tasks": ep_row.get("tasks", []),
                    "length": int(n),
                }
            )

            _copy_episode_videos(src, MERGE_ROOT, old_ep, new_ep, chunks_size)

            global_ep += 1
            global_frame += n

    with open(MERGE_ROOT / "meta" / "episodes.jsonl", "w", encoding="utf-8") as f:
        for row in merged_episodes_lines:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(MERGE_ROOT / "meta" / "episodes_stats.jsonl", "w", encoding="utf-8") as f:
        for row in merged_stats_lines:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    shutil.copy2(SOURCE_ROOTS[0] / "meta" / "tasks.jsonl", MERGE_ROOT / "meta" / "tasks.jsonl")
    for extra in ("modality.json",):
        p0 = SOURCE_ROOTS[0] / "meta" / extra
        if p0.is_file():
            shutil.copy2(p0, MERGE_ROOT / "meta" / extra)

    new_info = copy.deepcopy(ref)
    new_info["total_episodes"] = int(global_ep)
    new_info["total_frames"] = int(global_frame)
    has_any_video = any((s / "videos").is_dir() for s in SOURCE_ROOTS)
    new_info["total_videos"] = int(global_ep) if has_any_video else 0
    new_info["total_chunks"] = (
        (global_ep + chunks_size - 1) // chunks_size if global_ep else 0
    )
    new_info["splits"] = {"train": f"0:{global_ep}"}
    write_json(new_info, MERGE_ROOT / "meta" / "info.json")

    stats_path_src = SOURCE_ROOTS[0] / STATS_PATH
    if stats_path_src.is_file():
        with open(stats_path_src, encoding="utf-8") as f:
            old_stats = json.load(f)
    else:
        old_stats = {}

    vec = _recompute_global_vector_stats(MERGE_ROOT, ["observation.state", "action"], state_dim)
    scal = _recompute_global_scalar_stats(
        MERGE_ROOT,
        ["timestamp", "frame_index", "episode_index", "index", "task_index"],
    )
    merged_stats = dict(old_stats) if old_stats else {}
    merged_stats.update(scal)
    merged_stats["observation.state"] = vec["observation.state"]
    merged_stats["action"] = vec["action"]
    write_json(merged_stats, MERGE_ROOT / STATS_PATH)

    for opt in (".gitattributes", "README.md"):
        p0 = SOURCE_ROOTS[0] / opt
        if p0.is_file():
            shutil.copy2(p0, MERGE_ROOT / opt)

    print("[ok] 병합 완료:", MERGE_ROOT)
    print(" total_episodes", global_ep, "total_frames", global_frame)


AttributeError: 'str' object has no attribute 'parent'

## 4) 요약 확인


In [ ]:
import json
from pathlib import Path

p = MERGE_ROOT / "meta" / "info.json"
with open(p, encoding="utf-8") as f:
    info = json.load(f)
print("MERGE_ROOT", MERGE_ROOT)
print("codebase_version", info.get("codebase_version"))
print("total_episodes", info.get("total_episodes"), "total_frames", info.get("total_frames"))
print("observation.state shape", info.get("features", {}).get("observation.state", {}).get("shape"))


## 5) Hugging Face 업로드 (선택)

v2.1 유지: `LeRobotDataset` 대신 `HfApi.upload_folder` 사용. `PUSH_TO_HUB = True` 로 켠 뒤 실행하세요.


In [ ]:
import json
from pathlib import Path

from huggingface_hub import HfApi
from lerobot.datasets.utils import create_lerobot_dataset_card

if not PUSH_TO_HUB:
    print("[skip] HF push (PUSH_TO_HUB=False)")
else:
    if not PUSH_REPO_ID or "/" not in PUSH_REPO_ID:
        raise ValueError("PUSH_REPO_ID 는 user-or-org/dataset-name 형식이어야 합니다.")
    if not MERGE_ROOT.is_dir():
        raise FileNotFoundError(f"병합 결과 없음: {MERGE_ROOT}")

    info_path = MERGE_ROOT / "meta" / "info.json"
    with open(info_path, encoding="utf-8") as f:
        hub_info = json.load(f)
    if hub_info.get("codebase_version") != "v2.1":
        print("[warn] codebase_version:", hub_info.get("codebase_version"))

    hub_api = HfApi()
    hub_api.create_repo(
        repo_id=PUSH_REPO_ID,
        private=PUSH_PRIVATE,
        repo_type="dataset",
        exist_ok=True,
    )
    print("업로드 중…", PUSH_REPO_ID, "<-", MERGE_ROOT)
    hub_api.upload_folder(
        repo_id=PUSH_REPO_ID,
        folder_path=str(MERGE_ROOT),
        repo_type="dataset",
        allow_patterns=None,
        ignore_patterns=["images/"],
    )
    card = create_lerobot_dataset_card(
        tags=None,
        dataset_info=hub_info,
        license="apache-2.0",
    )
    card.push_to_hub(repo_id=PUSH_REPO_ID, repo_type="dataset")
    print("[ok] Hub 업로드 완료 (v2.1):", PUSH_REPO_ID)
